# 04. Hand checks of the metric functions on the real data

**Purpose.** Show that the formulas in `scripts/metrics.py` are right on the actual prices, using methods that do not share code with them: arithmetic done by hand on the raw prices, plain-Python loops, a brute-force search, and the same metrics recomputed inside PostgreSQL. This is the "validation" line of each metric in `metric_definitions.md` and plan item 5. The tiny hand-worked examples (answers worked out on paper) are in `tests/test_metrics.py` and run without the database: `python -I tests/test_metrics.py`.

**What is checked.**
- **M1** simple return: three specific days worked by hand from two prices each, including the BBCA split day (2021-10-13) and the BMRI ex-dividend day (2026-05-11).
- **M2** cumulative return over one regime: a plain loop of `(1 + r)`, the price ratio, and values already confirmed in earlier results.
- **M3** annualized return: over a one-year window it must be close to the cumulative return.
- **M4, M6, M2** against SQL: annualized volatility (1e-10), Sharpe ratio and cumulative return (1e-9), for each ticker over the full period and each regime.
- **M6** Sharpe ratio: a 20-day sample done by hand, and the rolling 252-day window size and first valid date.
- **M9** maximum drawdown: a brute-force search over every pair of days, a hand check of the peak and trough prices, and a SQL window-function calculation (1e-10).

**How to run.** Select the project `.venv` as the kernel, then use *Restart* and *Run All*. The notebook asks for the PostgreSQL user and password at the first data cell. It only reads from the database. Every check must print PASS.

In [1]:
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "scripts"))
try:
    import db_access
    import metrics
except ImportError as exc:
    raise ImportError(
        f"{exc}\nThis kernel is {sys.executable}. Select the project .venv kernel "
        "(top right of the notebook, Select Kernel > Python Environments > .venv), then Restart and Run All."
    ) from exc

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 30)
print("project root:", ROOT)

project root: d:\Data Analyst\Portfolio


## 1. Load the data

Four reads, all `SELECT` only. The prices and the policy rate rebuild the returns in Python. `load_metrics_sql` and `load_mdd_sql` compute cumulative return, volatility, Sharpe ratio and maximum drawdown inside PostgreSQL (`exp(sum(ln(1 + r)))`, `stddev_samp`, a window `max()`), without using `metrics.py`.

In [2]:
conn = db_access.connect()
try:
    prices = db_access.load_prices(conn)
    sql = db_access.load_excess_returns(conn)
    sql_metrics = db_access.load_metrics_sql(conn)
    sql_mdd = db_access.load_mdd_sql(conn)
finally:
    conn.close()

print("prices rows:", len(prices), "| SQL view rows:", len(sql),
      "| SQL metric rows:", len(sql_metrics), "| SQL drawdown rows:", len(sql_mdd))

prices rows: 5536 | SQL view rows: 5524 | SQL metric rows: 20 | SQL drawdown rows: 20


## 2. Settings and helpers

The expected values below are ones already confirmed elsewhere in the project: the three day returns from the SQL validation and the notebook 02 results, and three regime cumulative returns from the notebook 02 summary table. Percentages were recorded to two decimals (BBCA's split-day return to one), so the tolerances are half a unit in the last recorded digit.

In [3]:
TICKERS = ["BBCA", "BBRI", "BMRI", "IHSG"]
FIRST_RETURN_DAY = pd.Timestamp("2021-01-04")
EXPECTED_DAYS = 1381
REGIME_DAYS = {"R1": 399, "R2": 499, "R3": 393, "Stub": 90}

# ticker, trade date, expected close return, tolerance, expected adjusted-close return (or None), note
HAND_DAYS = [
    ("BBCA", "2021-10-13", 0.028, 0.0005, None, "stock split day (close is split-adjusted)"),
    ("BMRI", "2026-05-11", -0.0821, 0.00005, -0.0007, "ex-dividend day (adjusted close barely moves)"),
    ("BBRI", "2026-06-09", 0.0772, 0.00005, None, "market-wide rebound"),
]
KNOWN_REGIME_CUM = {("R2", "BBCA"): 0.3125, ("R3", "BBCA"): -0.4333, ("R1", "BMRI"): 0.3478}
KNOWN_TOL = 5e-5

TOL_COMPOUND = 1e-9
TOL_VOL = 1e-10
TOL_SHARPE = 1e-9
TOL_MDD = 1e-10

results = []


def check(name, ok, detail=""):
    results.append(bool(ok))
    print(f"{'PASS' if ok else 'FAIL'}  {name}" + (f"  [{detail}]" if detail else ""))


def sharpe_by_hand(values):
    """Annualized Sharpe ratio with plain Python only: mean / sample sd * sqrt(252)."""
    n = len(values)
    mean = sum(values) / n
    sd = math.sqrt(sum((v - mean) ** 2 for v in values) / (n - 1))
    return mean / sd * math.sqrt(252)

## 3. Returns and periods

The same construction as notebooks 02 and 03: simple returns from `close`, daily risk-free rate = policy rate / 100 / 252, excess return = return minus that rate.

In [4]:
close = db_access.to_wide(prices, "close")[TICKERS]
adj = db_access.to_wide(prices, "adj_close")[TICKERS]
ret = metrics.simple_returns(close, FIRST_RETURN_DAY)

rate = sql.drop_duplicates("trade_date").set_index("trade_date")["rate_pct"].sort_index()
excess = metrics.excess_return(ret, metrics.daily_risk_free(rate))

labels = metrics.assign_regime(ret.index)
period_days = {"Full": ret.index}
period_days.update({name: ret.index[labels == name] for name in REGIME_DAYS})

check(f"{EXPECTED_DAYS} return days, no missing values", len(ret) == EXPECTED_DAYS and not excess.isna().any().any())
check("regime day counts", all(len(period_days[r]) == n for r, n in REGIME_DAYS.items()), str(REGIME_DAYS))


def price_path(ticker, days):
    """Closes from the day before the first return day through the last return day."""
    pos = close.index.get_loc(days[0])
    return close[ticker].loc[close.index[pos - 1]:days[-1]]

PASS  1381 return days, no missing values
PASS  regime day counts  [{'R1': 399, 'R2': 499, 'R3': 393, 'Stub': 90}]


## 4. M1: three days worked by hand

For each day the return is worked out from the raw prices with one division, `P_t / P_(t-1) - 1`, and compared with `metrics.simple_returns` and with the value confirmed earlier. The BBCA split day checks that the stored `close` is split-adjusted (a raw price series would show a drop of about half). The BMRI ex-dividend day shows the price return and the adjusted-close return side by side.

In [5]:
rows = []
for ticker, day, expected, tol, expected_adj, note in HAND_DAYS:
    d = pd.Timestamp(day)
    pos = close.index.get_loc(d)
    p0, p1 = float(close[ticker].iloc[pos - 1]), float(close[ticker].iloc[pos])
    a0, a1 = float(adj[ticker].iloc[pos - 1]), float(adj[ticker].iloc[pos])
    by_hand, adj_by_hand, from_metrics = p1 / p0 - 1, a1 / a0 - 1, float(ret.loc[d, ticker])
    print(f"{ticker} {day}: {p1} / {p0} - 1 = {by_hand:.6f}   ({note})")
    check(f"M1 {ticker} {day}: by hand equals metrics.simple_returns", abs(by_hand - from_metrics) < 1e-12,
          f"{100 * by_hand:.4f}%")
    check(f"M1 {ticker} {day}: matches the value confirmed earlier", abs(by_hand - expected) < tol,
          f"expected {100 * expected:.2f}%")
    if expected_adj is not None:
        check(f"M1 {ticker} {day}: adjusted-close return matches", abs(adj_by_hand - expected_adj) < 5e-5,
              f"{100 * adj_by_hand:.4f}% against expected {100 * expected_adj:.2f}%")
    rows.append({"ticker": ticker, "date": day, "prior_close": p0, "close": p1,
                 "return_pct": round(100 * by_hand, 4), "adj_return_pct": round(100 * adj_by_hand, 4)})
pd.DataFrame(rows)

BBCA 2021-10-13: 7525.0 / 7320.0 - 1 = 0.028005   (stock split day (close is split-adjusted))
PASS  M1 BBCA 2021-10-13: by hand equals metrics.simple_returns  [2.8005%]
PASS  M1 BBCA 2021-10-13: matches the value confirmed earlier  [expected 2.80%]
BMRI 2026-05-11: 4250.0 / 4630.0 - 1 = -0.082073   (ex-dividend day (adjusted close barely moves))
PASS  M1 BMRI 2026-05-11: by hand equals metrics.simple_returns  [-8.2073%]
PASS  M1 BMRI 2026-05-11: matches the value confirmed earlier  [expected -8.21%]
PASS  M1 BMRI 2026-05-11: adjusted-close return matches  [-0.0715% against expected -0.07%]
BBRI 2026-06-09: 2790.0 / 2590.0 - 1 = 0.077220   (market-wide rebound)
PASS  M1 BBRI 2026-06-09: by hand equals metrics.simple_returns  [7.7220%]
PASS  M1 BBRI 2026-06-09: matches the value confirmed earlier  [expected 7.72%]


,ticker,date,prior_close,close,return_pct,adj_return_pct
0,BBCA,2021-10-13,7320.0,7525.0,2.8005,2.8005
1,BMRI,2026-05-11,4630.0,4250.0,-8.2073,-0.0715
2,BBRI,2026-06-09,2590.0,2790.0,7.7220,7.7220


## 5. M2: cumulative return over one regime

Three checks per regime. A plain Python loop multiplies `(1 + r)` day by day. The compounded value must equal the price ratio (last close over the close before the first day, minus one), which proves the daily returns chain without a gap. And it must match the figure already confirmed in the notebook 02 summary.

In [6]:
rows = []
for (period, ticker), expected in KNOWN_REGIME_CUM.items():
    days = period_days[period]
    product = 1.0
    for r in ret.loc[days, ticker].to_list():
        product *= 1 + r
    by_hand = product - 1
    start = close.index[close.index.get_loc(days[0]) - 1]
    ratio = float(close.loc[days[-1], ticker] / close.loc[start, ticker] - 1)
    fn = float(metrics.cumulative_return(ret.loc[days, ticker]))
    check(f"M2 {ticker} {period}: loop equals metrics.cumulative_return", abs(by_hand - fn) < 1e-12)
    check(f"M2 {ticker} {period}: compounded returns equal the price ratio", abs(by_hand - ratio) < TOL_COMPOUND,
          f"{100 * by_hand:.4f}%")
    check(f"M2 {ticker} {period}: matches the value confirmed earlier", abs(by_hand - expected) < KNOWN_TOL,
          f"expected {100 * expected:.2f}%")
    rows.append({"period": period, "ticker": ticker, "days": len(days), "start_close": float(close.loc[start, ticker]),
                 "end_close": float(close.loc[days[-1], ticker]), "cum_return_pct": round(100 * by_hand, 4)})
pd.DataFrame(rows)

PASS  M2 BBCA R2: loop equals metrics.cumulative_return
PASS  M2 BBCA R2: compounded returns equal the price ratio  [31.2500%]
PASS  M2 BBCA R2: matches the value confirmed earlier  [expected 31.25%]
PASS  M2 BBCA R3: loop equals metrics.cumulative_return
PASS  M2 BBCA R3: compounded returns equal the price ratio  [-43.3333%]
PASS  M2 BBCA R3: matches the value confirmed earlier  [expected -43.33%]
PASS  M2 BMRI R1: loop equals metrics.cumulative_return
PASS  M2 BMRI R1: compounded returns equal the price ratio  [34.7826%]
PASS  M2 BMRI R1: matches the value confirmed earlier  [expected 34.78%]


,period,ticker,days,start_close,end_close,cum_return_pct
0,R2,BBCA,499,8000.0,10500.0,31.2500
1,R3,BBCA,393,10500.0,5950.0,-43.3333
2,R1,BMRI,399,3162.5,4262.5,34.7826


## 6. M3: annualized return over a one-year window

For a window of about one calendar year, the annualized return must be close to the cumulative return, because the exponent `365.25 / D` is close to 1. The window runs from the last close of 2021 to the last close of 2022. The trading-day version (`252 / n`) is shown for comparison: this window has about 240 trading days, not 252, so the two versions differ by a few percent of the return. That gap is the known cost of the 252-day convention and is not an error.

In [7]:
w_start = close.index[close.index <= "2021-12-31"].max()
w_end = close.index[close.index <= "2022-12-31"].max()
w_days = ret.index[(ret.index > w_start) & (ret.index <= w_end)]
calendar_days = (w_end - w_start).days
print(f"window {w_start.date()} to {w_end.date()}: {calendar_days} calendar days, {len(w_days)} return days")

rows = []
for t in TICKERS:
    cr_prices = float(close.loc[w_end, t] / close.loc[w_start, t] - 1)
    cr_returns = float(metrics.cumulative_return(ret.loc[w_days, t]))
    ar = float(metrics.annualized_return(cr_returns, calendar_days))
    ar_252 = float(metrics.annualized_return_trading(cr_returns, len(w_days)))
    check(f"M3 {t}: window cumulative return equals the price ratio", abs(cr_prices - cr_returns) < TOL_COMPOUND)
    check(f"M3 {t}: one-year annualized return is close to the cumulative return", abs(ar - cr_returns) < 0.01,
          f"CR {100 * cr_returns:.2f}%, AR {100 * ar:.2f}%")
    rows.append({"ticker": t, "cum_return_pct": round(100 * cr_returns, 3), "ann_return_pct": round(100 * ar, 3),
                 "ann_return_252_pct": round(100 * ar_252, 3)})
pd.DataFrame(rows)

window 2021-12-30 to 2022-12-30: 365 calendar days, 246 return days
PASS  M3 BBCA: window cumulative return equals the price ratio
PASS  M3 BBCA: one-year annualized return is close to the cumulative return  [CR 17.12%, AR 17.14%]
PASS  M3 BBRI: window cumulative return equals the price ratio
PASS  M3 BBRI: one-year annualized return is close to the cumulative return  [CR 20.19%, AR 20.21%]
PASS  M3 BMRI: window cumulative return equals the price ratio
PASS  M3 BMRI: one-year annualized return is close to the cumulative return  [CR 41.28%, AR 41.31%]
PASS  M3 IHSG: window cumulative return equals the price ratio
PASS  M3 IHSG: one-year annualized return is close to the cumulative return  [CR 4.09%, AR 4.09%]


,ticker,cum_return_pct,ann_return_pct,ann_return_252_pct
0,BBCA,17.123,17.136,17.576
1,BBRI,20.195,20.210,20.735
2,BMRI,41.281,41.315,42.477
3,IHSG,4.089,4.092,4.191


## 7. M2, M4 and M6 against SQL

The same metrics recomputed in PostgreSQL for every ticker over the full period and each regime. Cumulative return uses `exp(sum(ln(1 + r))) - 1`, volatility uses `stddev_samp * sqrt(252)`, and the Sharpe ratio is `avg(e) / stddev_samp(e) * sqrt(252)` on the excess returns of the SQL view. Tolerances: volatility 1e-10, cumulative return and Sharpe 1e-9.

In [8]:
rows = []
for period, days in period_days.items():
    for t in TICKERS:
        rows.append({"ticker": t, "period": period, "n": len(days),
                     "cum_return": metrics.cumulative_return(ret.loc[days, t]),
                     "ann_vol": metrics.annualized_volatility(ret.loc[days, t]),
                     "sharpe": metrics.sharpe_ratio(excess.loc[days, t])})
py = pd.DataFrame(rows).set_index(["ticker", "period"]).sort_index()
sq = sql_metrics.set_index(["ticker", "period"]).sort_index()

check("SQL returns the same 20 ticker-period rows", py.index.equals(sq.index), f"{len(sq)} rows")
check("return-day counts equal", (py["n"] == sq["n"]).all())
for col, tol, label in (("cum_return", TOL_COMPOUND, "M2 cumulative return"), ("ann_vol", TOL_VOL, "M4 annualized volatility"),
                        ("sharpe", TOL_SHARPE, "M6 Sharpe ratio")):
    diff = (py[col] - sq[col]).abs().max()
    check(f"{label}: Python equals SQL", diff < tol, f"max abs diff {diff:.2e}")

PASS  SQL returns the same 20 ticker-period rows  [20 rows]
PASS  return-day counts equal
PASS  M2 cumulative return: Python equals SQL  [max abs diff 6.88e-15]
PASS  M4 annualized volatility: Python equals SQL  [max abs diff 2.22e-16]
PASS  M6 Sharpe ratio: Python equals SQL  [max abs diff 8.40e-15]


## 8. M6: a 20-day Sharpe ratio by hand, and the rolling window

The first 20 excess returns of each instrument go through a plain Python calculation (mean, sample standard deviation with `n - 1`, ratio, times the square root of 252), with no pandas or numpy. The 20 BBCA values are printed in percent so the same arithmetic can be repeated in a spreadsheet.

For the rolling Sharpe ratio, the value on one chosen day must equal the Sharpe ratio of exactly the last 252 return days, and must differ from the value for a 251-day window. The first value must be on the 252nd return day.

In [9]:
sample = excess.iloc[:20]
rows = []
for t in TICKERS:
    values = sample[t].to_list()
    by_hand = sharpe_by_hand(values)
    fn = float(metrics.sharpe_ratio(sample[t]))
    check(f"M6 {t}: 20-day Sharpe by hand equals metrics.sharpe_ratio", abs(by_hand - fn) < 1e-12, f"{by_hand:.4f}")
    rows.append({"ticker": t, "mean_pct": round(100 * sum(values) / 20, 5), "sharpe_20d": round(by_hand, 4)})
print(pd.DataFrame(rows).to_string(index=False))
print("\nBBCA excess returns for the first 20 days (percent):")
print((100 * sample["BBCA"]).round(5).to_string())

PASS  M6 BBCA: 20-day Sharpe by hand equals metrics.sharpe_ratio  [-0.0402]
PASS  M6 BBRI: 20-day Sharpe by hand equals metrics.sharpe_ratio  [0.2195]
PASS  M6 BMRI: 20-day Sharpe by hand equals metrics.sharpe_ratio  [1.2404]
PASS  M6 IHSG: 20-day Sharpe by hand equals metrics.sharpe_ratio  [-1.2089]
ticker  mean_pct  sharpe_20d
  BBCA  -0.00487     -0.0402
  BBRI   0.04302      0.2195
  BMRI   0.21538      1.2404
  IHSG  -0.10447     -1.2089

BBCA excess returns for the first 20 days (percent):
trade_date
2021-01-04    0.94524
2021-01-05    3.71592
2021-01-06   -2.06001
2021-01-07    0.27310
2021-01-08    1.20551
2021-01-11    4.16952
2021-01-12   -2.53360
2021-01-13   -0.57354
2021-01-14   -1.41938
2021-01-15   -0.94081
2021-01-18    2.35751
2021-01-19   -0.36600
2021-01-20   -0.01488
2021-01-21   -0.29677
2021-01-22    0.05579
2021-01-25   -0.65047
2021-01-26   -3.07103
2021-01-27   -0.45476
2021-01-28    1.60515
2021-01-29   -2.04387


In [10]:
rs = metrics.rolling_sharpe(excess, window=252)
pos = 700
for t in TICKERS:
    window = excess[t].iloc[pos - 251:pos + 1].to_list()
    short = excess[t].iloc[pos - 250:pos + 1].to_list()
    check(f"M6 {t}: rolling value on {excess.index[pos].date()} equals a 252-day hand calculation",
          abs(rs[t].iloc[pos] - sharpe_by_hand(window)) < 1e-12 and len(window) == 252)
    check(f"M6 {t}: it differs from a 251-day window", abs(rs[t].iloc[pos] - sharpe_by_hand(short)) > 1e-6)

check("rolling Sharpe: first value on the 252nd return day",
      rs.first_valid_index() == ret.index[251] and rs.iloc[:251].isna().all().all(), f"{rs.first_valid_index().date()}")
check("rolling Sharpe: one value for every day from the 252nd on", int(rs["BBCA"].notna().sum()) == EXPECTED_DAYS - 251)

PASS  M6 BBCA: rolling value on 2023-11-14 equals a 252-day hand calculation
PASS  M6 BBCA: it differs from a 251-day window
PASS  M6 BBRI: rolling value on 2023-11-14 equals a 252-day hand calculation
PASS  M6 BBRI: it differs from a 251-day window
PASS  M6 BMRI: rolling value on 2023-11-14 equals a 252-day hand calculation
PASS  M6 BMRI: it differs from a 251-day window
PASS  M6 IHSG: rolling value on 2023-11-14 equals a 252-day hand calculation
PASS  M6 IHSG: it differs from a 251-day window
PASS  rolling Sharpe: first value on the 252nd return day  [2022-01-07]
PASS  rolling Sharpe: one value for every day from the 252nd on


## 9. M9: maximum drawdown three ways

1. **Brute force.** The maximum drawdown is the worst `P_j / P_i - 1` over every pair of days with `i` before `j`. This is checked for every ticker and period against `metrics.max_drawdown`. It uses no running maximum, so it is a different method.
2. **By hand on the full period.** For each ticker the peak and trough prices on the reported dates are taken from the raw closes: the drawdown must equal `trough / peak - 1`, and the peak must be the highest close up to the trough date.
3. **SQL.** The same drawdown with a window `max()` in PostgreSQL, per ticker and period (tolerance 1e-10).

Each regime's path starts at the close before its first return day, so the running maximum starts at the buy price.

In [11]:
rows = []
for period, days in period_days.items():
    for t in TICKERS:
        path = price_path(t, days)
        p = path.to_numpy()
        brute = float(np.triu(p[None, :] / p[:, None] - 1, k=1).min())
        fn = metrics.max_drawdown(path)
        rows.append({"ticker": t, "period": period, "mdd": fn["mdd"], "brute_force": brute,
                     "peak_date": fn["peak_date"].date(), "trough_date": fn["trough_date"].date(),
                     "peak_close": float(path.loc[fn["peak_date"]]), "trough_close": float(path.loc[fn["trough_date"]])})
mdd = pd.DataFrame(rows).set_index(["ticker", "period"]).sort_index()

d_brute = (mdd["mdd"] - mdd["brute_force"]).abs().max()
check("M9: metrics.max_drawdown equals a brute-force search over all pairs of days", d_brute < 1e-12, f"max abs diff {d_brute:.2e}")

full = mdd.xs("Full", level="period")
by_hand = full["trough_close"] / full["peak_close"] - 1
check("M9 full period: trough / peak - 1 equals the reported drawdown", (by_hand - full["mdd"]).abs().max() < 1e-12)
check("M9 full period: the peak is the highest close up to the trough date",
      all(abs(price_path(t, ret.index).loc[:pd.Timestamp(full.loc[t, "trough_date"])].max() - full.loc[t, "peak_close"]) < 1e-9
          for t in TICKERS))

sqm = sql_mdd.set_index(["ticker", "period"]).sort_index()
check("SQL returns the same 20 ticker-period drawdowns", mdd.index.equals(sqm.index), f"{len(sqm)} rows")
d_sql = (mdd["mdd"] - sqm["mdd"]).abs().max()
check("M9: Python equals the SQL window-function drawdown", d_sql < TOL_MDD, f"max abs diff {d_sql:.2e}")
(100 * mdd[["mdd"]]).round(2).rename(columns={"mdd": "mdd_pct"}).join(mdd[["peak_date", "trough_date", "peak_close", "trough_close"]])

PASS  M9: metrics.max_drawdown equals a brute-force search over all pairs of days  [max abs diff 0.00e+00]
PASS  M9 full period: trough / peak - 1 equals the reported drawdown
PASS  M9 full period: the peak is the highest close up to the trough date
PASS  SQL returns the same 20 ticker-period drawdowns  [20 rows]
PASS  M9: Python equals the SQL window-function drawdown  [max abs diff 0.00e+00]


mdd_pct   peak_date trough_date    peak_close  trough_close
ticker period                                                             
BBCA   Full     -55.71  2024-09-23  2026-06-08  10950.000000   4850.000000
       R1       -18.86  2021-01-11  2021-08-02   7345.000000   5960.000000
       R2       -13.44  2022-11-30  2023-01-13   9300.000000   8050.000000
       R3       -46.58  2024-09-23  2026-04-30  10950.000000   5850.000000
       Stub     -20.49  2026-05-25  2026-06-08   6100.000000   4850.000000
BBRI   Full     -59.53  2024-03-13  2026-06-08   6400.000000   2590.000000
       R1       -24.13  2021-01-20  2021-07-14   4445.377930   3372.668945
       R2       -35.94  2024-03-13  2024-06-19   6400.000000   4100.000000
       R3       -45.88  2024-09-24  2026-04-30   5525.000000   2990.000000
       Stub     -18.30  2026-05-25  2026-06-08   3170.000000   2590.000000
BMRI   Full     -50.20  2024-09-23  2026-06-08   7450.000000   3710.000000
       R1       -23.39  2021-01-20  2021-05-20   3687.500000   2825.000000
       R2       -22.30  2024-03-14  2024-06-14   7400.000000   5750.000000
       R3       -45.64  2024-09-23  2025-10-15   7450.000000   4050.000000
       Stub     -15.33  2026-06-15  2026-07-01   4500.000000   3810.000000
IHSG   Full     -41.52  2026-01-20  2026-06-08   9134.700195   5342.137207
       R1       -10.48  2021-01-13  2021-05-19   6435.205078   5760.583984
       R2       -10.28  2022-09-13  2023-03-16   7318.016113   6565.728027
       R3       -30.26  2026-01-20  2026-05-19   9134.700195   6370.679199
       Stub     -16.14  2026-05-19  2026-06-08   6370.679199   5342.137207

## 10. Summary

Every check above must print PASS. Together with the offline tests in `tests/test_metrics.py` (hand-worked examples for M1 to M9, the bootstrap, CAPM, Dimson beta and the regime boundaries), this closes the hand-check item of the task plan.

In [12]:
n_fail = results.count(False)
print(f"{len(results) - n_fail} passed, {n_fail} failed")
assert n_fail == 0, "At least one hand check failed. Investigate before relying on the metric functions."

50 passed, 0 failed
